### Random Forest Regression

#### Used Car Price Prediction

##### 1) Problem statement.

* This dataset comprises used cars sold on cardehko.com in India as well as important features of these cars.
* If user can predict the price of the car based on input features.
* Prediction results can be used to give new seller the price suggestion based on market condition.

##### 2) Data Collection.
* The Dataset is collected from scrapping from cardheko webiste
* The data consists of 13 column and 15411 rows.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
%matplotlib inline


In [5]:
df=pd.read_csv("data/cardekho_imputated.csv",index_col=[0])
display(df)

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19537,Hyundai i10,Hyundai,i10,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5,250000
19540,Maruti Ertiga,Maruti,Ertiga,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7,925000
19541,Skoda Rapid,Skoda,Rapid,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5,425000
19542,Mahindra XUV500,Mahindra,XUV500,5,3800000,Dealer,Diesel,Manual,16.00,2179,140.00,7,1225000


In [6]:
## check for null values
## check for features as nan value
## 0===> downward 1----> sideward
df.isnull().sum(axis=0)

car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [7]:
df.drop("car_name",inplace=True,axis=1)
df.drop("brand",inplace=True,axis=1)
df['model'].unique()

array(['Alto', 'Grand', 'i20', 'Ecosport', 'Wagon R', 'i10', 'Venue',
       'Swift', 'Verna', 'Duster', 'Cooper', 'Ciaz', 'C-Class', 'Innova',
       'Baleno', 'Swift Dzire', 'Vento', 'Creta', 'City', 'Bolero',
       'Fortuner', 'KWID', 'Amaze', 'Santro', 'XUV500', 'KUV100', 'Ignis',
       'RediGO', 'Scorpio', 'Marazzo', 'Aspire', 'Figo', 'Vitara',
       'Tiago', 'Polo', 'Seltos', 'Celerio', 'GO', '5', 'CR-V',
       'Endeavour', 'KUV', 'Jazz', '3', 'A4', 'Tigor', 'Ertiga', 'Safari',
       'Thar', 'Hexa', 'Rover', 'Eeco', 'A6', 'E-Class', 'Q7', 'Z4', '6',
       'XF', 'X5', 'Hector', 'Civic', 'D-Max', 'Cayenne', 'X1', 'Rapid',
       'Freestyle', 'Superb', 'Nexon', 'XUV300', 'Dzire VXI', 'S90',
       'WR-V', 'XL6', 'Triber', 'ES', 'Wrangler', 'Camry', 'Elantra',
       'Yaris', 'GL-Class', '7', 'S-Presso', 'Dzire LXI', 'Aura', 'XC',
       'Ghibli', 'Continental', 'CR', 'Kicks', 'S-Class', 'Tucson',
       'Harrier', 'X3', 'Octavia', 'Compass', 'CLS', 'redi-GO', 'Glanza',
       

In [8]:
## Getting All Features details 
num_features=[feature for feature in df.columns if df[feature].dtype != 'O']
print(f"Numerical Features : {len(num_features)}")
cat_features=[feature for feature in df.columns if df[feature].dtype == 'O']
print(f"Categorical Features : {len(cat_features)}")
discrete_features=[feature for feature in df.columns if len(df[feature].unique()) < 7]
print(f"Categorical Features : {len(discrete_features)}")
continuous_features=[feature for feature in df.columns if len(df[feature].unique()) > 7]
print(f"Categorical Features : {len(continuous_features)}")

Numerical Features : 7
Categorical Features : 4
Categorical Features : 3
Categorical Features : 8


In [9]:
from sklearn.model_selection import train_test_split
X=df.drop(["selling_price"],axis=1)
y=df["selling_price"]

In [10]:
X.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


In [11]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3)

In [12]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
X_train['model']=le.fit_transform(X_train['model'])

#important earlier it was sying ghost is new key
#Ordinal encode should be used when order matters or it is tree based model eg school< college < phd
def safe_label_encode(le, series):
    return series.map(lambda x: le.transform([x])[0] if x in le.classes_ else -1)

X_test['model'] = safe_label_encode(le, X_test['model'])
print(X_train.shape)

(10787, 10)


In [13]:
num_features=X_train.select_dtypes(exclude='object').columns
one_hot_column=['seller_type','fuel_type','transmission_type']
lable_encoder_column=['model']
#Feature Encoding and Scaling
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

#lable encoder can not be used in columnTransformer as it expects only one column 
transformer=ColumnTransformer([
    ("OneHotEncoder",OneHotEncoder(drop='first'),one_hot_column),
     ("Standard Scaler",StandardScaler(),num_features),
],remainder='passthrough')
X_train=transformer.fit_transform(X_train)
display(X_train)
X_test=transformer.transform(X_test)

array([[ 1.        ,  0.        ,  0.        , ...,  0.01678146,
        -0.01304585, -0.4080075 ],
       [ 0.        ,  0.        ,  0.        , ..., -0.55410643,
        -0.50652931, -0.4080075 ],
       [ 0.        ,  0.        ,  0.        , ..., -0.55603511,
        -0.7146811 , -0.4080075 ],
       ...,
       [ 0.        ,  0.        ,  0.        , ..., -0.55410643,
        -0.36386347, -0.4080075 ],
       [ 0.        ,  0.        ,  1.        , ...,  1.33985272,
         0.92246782,  2.05971156],
       [ 1.        ,  0.        ,  0.        , ..., -0.93598414,
        -0.78484462, -0.4080075 ]], shape=(10787, 14))

In [14]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge , Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error

In [15]:
#Function to evaluate model
def evaluate_model(true,predicted):
    mae=mean_absolute_error(true,predicted)
    mse=mean_squared_error(true,predicted)
    rmse=np.sqrt(mean_squared_error(true,predicted))
    r2_square=r2_score(true,predicted)
    return mae,mse,rmse,r2_square


model={
    "Linear Regression" : LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "Decision Tree": DecisionTreeRegressor(),
    "KNN" : KNeighborsRegressor(),
    "Random Forest": RandomForestRegressor()
}
for key, value in model.items():
    value.fit(X_train,y_train)
    y_train_pred=value.predict(X_train)
    y_test_pred=value.predict(X_test)
    mae_train,mse_train,rmse_train,r2_square_train=evaluate_model(y_train,y_train_pred)
    mae_test,mse_test,rmse_test,r2_square_test=evaluate_model(y_test,y_test_pred)

    print(f"========{key}=======")
    print(f"========Model Performance on Train Dataset=======")
    print(f"mae={mae_train}")
    print(f"mse={mse_train}")
    print(f"rmse={rmse_train}")
    print(f"r2_sqr={r2_square_train}")

    print(f"========Model Performance on test Dataset=======")
    print(f"mae={mae_test}")
    print(f"mse={mse_test}")
    print(f"rmse={rmse_test}")
    print(f"r2_sqr={r2_square_test}")




========Linear Regression=======
========Model Performance on Train Dataset=======
mae=273646.14890050824
mse=326000541885.5299
rmse=570964.5714801663
r2_sqr=0.6179198362680693
========Model Performance on test Dataset=======
mae=276309.95132672956
mse=227459544161.12375
rmse=476927.1895804681
r2_sqr=0.6623722708163926
========Lasso=======
========Model Performance on Train Dataset=======
mae=273644.8873333594
mse=326000548085.6647
rmse=570964.576909693
r2_sqr=0.6179198290013683
========Model Performance on test Dataset=======
mae=276306.5830493309
mse=227458428047.04318
rmse=476926.01946952235
r2_sqr=0.6623739275112739
========Ridge=======
========Model Performance on Train Dataset=======
mae=273611.49144287925
mse=326001721829.6169
rmse=570965.6047693389
r2_sqr=0.6179184533463499
========Model Performance on test Dataset=======
mae=276251.6012232798
mse=227444011104.75552
rmse=476910.90478700056
r2_sqr=0.6623953271826059
========Decision Tree=======
========Model Performance on Train

#### Hyper Parameter Tuning 

In [18]:
knn_param={"n_neighbors":[2,3,4,9,10,50,39]}
rf_param={"max_depth":[5,8,15,None,10],
          "max_features": [5,7,"auto",9],
          "min_samples_split": [2,8,15,20],
          "n_estimators":[100,200,300,350,400,100]}


ramdom_cv_model=[("knn",KNeighborsRegressor(),knn_param),
                 ("RF",RandomForestRegressor(),rf_param)]

model_params={}
from sklearn.model_selection import RandomizedSearchCV
for name,model,params in ramdom_cv_model:
    random=RandomizedSearchCV(estimator=model,
                              param_distributions=params,
                              n_iter=100,
                              cv=3,
                              verbose=2,
                              n_jobs=-1)
    
    random.fit(X_train,y_train)
    model_params[name]=random.best_params_
for model_name in model_params:
    print(f"-----------------Best Params for model {model_name} ----------------")    
    print(model_params[model_name])

Fitting 3 folds for each of 7 candidates, totalling 21 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
-----------------Best Params for model knn ----------------
{'n_neighbors': 2}
-----------------Best Params for model RF ----------------
{'n_estimators': 200, 'min_samples_split': 2, 'max_features': 5, 'max_depth': 15}
